Description:

Analyze the relationship between the number of primary care appointments per head of population in sub-ICBs within SNEE (Suffolk and North East Essex) and staffing levels per head of population. The goal is to use the GP Patient list, the most recent appointments dataset, and the staffing dataset from NHSE (all currently available in the catalog) to investigate how the patient-to-GP ratio correlates with appointments per head specifically in the context of SNEE compared to broader English sub-ICBs.

Exam question/Objective:

Determine if staffing levels per head of population significantly affect the number of primary care appointments per head in sub-ICBs within SNEE. Additionally, assess whether the patient-to-GP ratio in SNEE reflects or deviates from the trends seen in other English sub-ICBs.

Stakeholder Request:

I want a report that details the relationship between the number of primary care appointments per head of population and staffing levels per head of population in sub-ICBs within SNEE. The report should highlight if variations in patient-to-GP ratios contribute to differences in appointments per head and whether these findings are consistent with or diverge from other English sub-ICBs.

Key Data Sources:

GP Patient list from NHSE
Most recent appointments dataset from NHSE
Staffing dataset from NHSE
Methodology/Approach:

Data extraction from NHSE catalog for GP Patient list, appointments data, and staffing data.
Clean and preprocess the datasets to ensure consistency.
Calculate patient-to-GP ratios and staffing levels per head of population.
Conduct initial exploratory data analysis (EDA) to understand distribution and correlations.
Apply statistical analysis (e.g., linear regression or correlation analysis) to assess relationships.
Compare results between SNEE sub-ICBs and other English sub-ICBs.
Visualize findings through charts and graphs.
Summarize insights in a report.
Outputs:

Branch in repo
Notebook
Markdown blog

## Library Imports

In [28]:
import os
from pathlib import Path
if 'notebooks' in str(Path.cwd()):
    os.chdir('..')

# Library imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from matplotlib.ticker import FuncFormatter
import datetime as dt
import pickle
from typing import Dict
from time import sleep
from datetime import timedelta

# project imports from src
from src.schemas import DataCatalog
from src.various_methods import PlotCounter, get_workingdays
from src import constants

# Importing SNEE styles
from sneeifstyles import mpl_style
mpl_style()

import warnings
warnings.filterwarnings("ignore")

## Initial Set-up

In [2]:
## Constants
SNEE_SUB_ICB = ['06L','06T','07K']
SNEE_SUB_ICB_NAMES = ['Ipswich & East Suffolk', 'North East Essex', 'West Suffolk']
NOTEBOOK_ALIAS = "Appointments"
GP_PATIENTS_LIST_CATALOG = 'Patients Registered at a GP practice, September 2024'
GP_APPOINTMENTS_CATALOG_NAME:str = 'Appointments in General Practice, August 2024'
GP_WORKFORCE_CATALOG:str = 'General Practice workforce'  # August/2024
GP_LIST_AGE_BANDS = constants.GP_LIST_AGE_BANDS 
GP_LIST_AGE_LABELS = constants.GP_LIST_AGE_LABELS 
SNEE_SUBICB_CODES = list(constants.ONS_CODES.keys())

# Loading the Data Catalog
catalog =  DataCatalog.load_from_yaml("data_catalog.yaml")

# Initializing the plotCounter object
plot_counter = PlotCounter(name=NOTEBOOK_ALIAS)

# set up output directories
for i in ['outputs/assumptions', 'outputs/plots', 'outputs/tables']:
    if not os.path.exists(i):
        os.makedirs(i)

## 1. GP list Loading and processing

In [3]:
def process_gp_list(df):
    """
    Args:
        df (pandas.DataFrame): GP LIST DataFrame. 
    Returns:
        pandas.DataFrame: 
    """
    df_ = df.copy()
    
    # Keeping only the total population both 'AGE_GROUP_5' and 'SEX' columns are 'ALL'
    df_ = df.loc[(df['AGE_GROUP_5'] == 'ALL') & (df['SEX'] == 'ALL')]
    
    # Filter rows to keep only those with ORG_TYPE as 'SUB_ICB_LOCATION_CODE'
    df_ = df_[df_['ORG_TYPE'].isin(['SUB_ICB_LOCATION_CODE'])].copy()
    
    df_ = df_.rename(columns={'ORG_CODE':'SUB_ICB_CODE'})

    # Dropping unused columns
    df_ = df_.drop(columns=['PUBLICATION','EXTRACT_DATE','ORG_TYPE','POSTCODE','SEX','AGE_GROUP_5']).reset_index(drop=True)
  
    return df_

In [4]:
gp_list_df = catalog.get_catalog_entry_by_name(GP_PATIENTS_LIST_CATALOG)
patients_df = gp_list_df.load()
patients_df = process_gp_list(patients_df)
patients_df.tail()

# # ICB level total
# icb_patients_df = patients_df[~patients_df['ICB'].isnull()].drop(columns={'SUB_ICB_LOCATION_CODE'}).reset_index(drop=True)
# icb_patients_df.tail()

,SUB_ICB_CODE,ONS_CODE,NUMBER_OF_PATIENTS
101,D9Y0V,E38000253,1730028
102,M1J4Y,E38000249,1140994
103,M2L0M,E38000257,534870
104,W2U3Z,E38000256,2899512
105,X2C4Y,E38000254,461152


## 2. Workforce Loading and processing

In [60]:
def process_workforce(df):
    """
    Args:
        df (pandas.DataFrame): Workforce DataFrame. 
    Returns:
        pandas.DataFrame: 
    """
    df_ = df.copy()
    
    # Dropping unused columns
    df_ = df_.drop(columns=['YEAR','Month','COMM_REGION_CODE','COMM_REGION_NAME','ICB_CODE','ICB_NAME','SUB_ICB_NAME','DATA_SOURCE','UNIQUE_IDENTIFIER','DETAILED_STAFF_ROLE',
                            'STAFF_ROLE','COUNTRY_QUALIFICATION_AREA','COUNTRY_QUALIFICATION_GROUP','AGE_BAND','AGE_YEARS','GENDER'])
    
    # Group by sub-icb and staff type
    df_ = df_.groupby(['SUB_ICB_CODE', 'STAFF_GROUP']).sum().round(2).reset_index()

    return df_

In [61]:
workforce_entry =  catalog.get_catalog_entry_by_name(GP_WORKFORCE_CATALOG)
workforce_df = workforce_entry.load()

# To match the names of icbs at the end
col = ['ICB_CODE','ICB_NAME','SUB_ICB_CODE','SUB_ICB_NAME']
icb_name_code_match = workforce_df[col].drop_duplicates().reset_index(drop=True)

workforce_df = process_workforce(workforce_df)
workforce_df

,SUB_ICB_CODE,STAFF_GROUP,FTE
0,00L,Admin/Non-Clinical,490.48
1,00L,Direct Patient Care,120.05
2,00L,GP,254.83
3,00L,Nurses,112.41
4,00N,Admin/Non-Clinical,214.06
...,...,...,...
420,W2U3Z,Nurses,374.31
421,X2C4Y,Admin/Non-Clinical,536.59
422,X2C4Y,Direct Patient Care,111.65
423,X2C4Y,GP,248.59


queation --> the FTE above is for one single day (should we multiply by number of working days and year to get full year worth of FTE)

## 3. Appointments Loading and processing (keeping 1 full year of appointments)

In [77]:
def process_appointments(df):
    """
    Args:
        df (pandas.DataFrame): Appointments DataFrame. 
    Returns:
        pandas.DataFrame: 
    """
    df_ = df.copy()
    
    # Convert the 'date_column' to datetime format
    df_['Appointment_Date'] = pd.to_datetime(df_['Appointment_Date'], format='%d%b%Y')

    # Calculate the date one year before the latest date
    one_year_ago = (df_['Appointment_Date'].max()) - timedelta(days=365)

    # Filter the DataFrame to get data from the last year
    df_ = df_[df_['Appointment_Date'] >= one_year_ago].reset_index(drop=True)
    
    # Dropping unused columns
    df_ = df_.drop(columns=['SUB_ICB_LOCATION_ONS_CODE','SUB_ICB_LOCATION_NAME','ICB_ONS_CODE','REGION_ONS_CODE','ACTUAL_DURATION'])
    
    # Group by 'SUB_ICB_LOCATION_CODE', and sum 'COUNT_OF_APPOINTMENTS' 
    df_ = df_.groupby(['SUB_ICB_LOCATION_CODE'])['COUNT_OF_APPOINTMENTS'].sum().reset_index().rename(columns={'SUB_ICB_LOCATION_CODE' : 'SUB_ICB_CODE'})
    

    return df_

In [78]:
appointments_catalog_entry = catalog.get_catalog_entry_by_name(GP_APPOINTMENTS_CATALOG_NAME)
appointments_df = appointments_catalog_entry.load()
appointments_df = process_appointments(appointments_df)
appointments_df

,SUB_ICB_CODE,COUNT_OF_APPOINTMENTS
0,00L,2089082
1,00N,712275
2,00P,1447505
3,00Q,793575
4,00R,960248
...,...,...
101,D9Y0V,9312912
102,M1J4Y,5231051
103,M2L0M,2774435
104,W2U3Z,13897197


## 4. Merging the datasets

In [72]:
appointments_vs_staffing_df = pd.merge(patients_df, appointments_df, on='SUB_ICB_CODE', how='inner')
appointments_vs_staffing_df

,SUB_ICB_CODE,ONS_CODE,NUMBER_OF_PATIENTS,COUNT_OF_APPOINTMENTS
0,00L,E38000130,340837,2089082
1,00N,E38000163,159989,712275
2,00P,E38000176,295091,1447505
3,00Q,E38000014,187096,793575
4,00R,E38000015,179482,960248
...,...,...,...,...
101,D9Y0V,E38000253,1730028,9312912
102,M1J4Y,E38000249,1140994,5231051
103,M2L0M,E38000257,534870,2774435
104,W2U3Z,E38000256,2899512,13897197


In [73]:
appointments_vs_staffing_df = pd.merge(patients_df, appointments_df, on='SUB_ICB_CODE', how='inner')
appointments_vs_staffing_df = pd.merge(appointments_vs_staffing_df, workforce_df, on='SUB_ICB_CODE', how='right')
appointments_vs_staffing_df

,SUB_ICB_CODE,ONS_CODE,NUMBER_OF_PATIENTS,COUNT_OF_APPOINTMENTS,STAFF_GROUP,FTE
0,00L,E38000130,340837.0,2089082.0,Admin/Non-Clinical,490.48
1,00L,E38000130,340837.0,2089082.0,Direct Patient Care,120.05
2,00L,E38000130,340837.0,2089082.0,GP,254.83
3,00L,E38000130,340837.0,2089082.0,Nurses,112.41
4,00N,E38000163,159989.0,712275.0,Admin/Non-Clinical,214.06
...,...,...,...,...,...,...
420,W2U3Z,E38000256,2899512.0,13897197.0,Nurses,374.31
421,X2C4Y,E38000254,461152.0,2972654.0,Admin/Non-Clinical,536.59
422,X2C4Y,E38000254,461152.0,2972654.0,Direct Patient Care,111.65
423,X2C4Y,E38000254,461152.0,2972654.0,GP,248.59


In [92]:
date_range_df = pd.DataFrame(pd.date_range(start=dt.date(year=2023,month=9,day=1), end=dt.date(year=2024,month=8,day=31), freq='MS')).rename(columns={0:'DATE'})
# Get working days/month
date_range_df['working_days'] = get_workingdays(date_range_df['DATE'].dt)

print(f'Total working days from 2023-09-01 to 2024-08-31 are: {date_range_df['working_days'].sum()}')

date_range_df

Total working days from 2023-09-01 to 2024-08-31 are: 255


,DATE,working_days
0,2023-09-01,21
1,2023-10-01,22
2,2023-11-01,22
3,2023-12-01,19
4,2024-01-01,22
5,2024-02-01,21
6,2024-03-01,20
7,2024-04-01,22
8,2024-05-01,21
9,2024-06-01,20
